# BirdCLEF 2026 - GPU training v50 (EfficientNetB0, full data)

## Purpose
Train ONE strong EfficientNetB0 model on Kaggle T4 GPU using ALL focal data + ALL soundscape labelled windows. Same architecture (B0, 64 mels, 626 frames) as our existing CPU pipeline so the resulting model is **compatible with the existing CPU submission notebook**.

Why this matters: we previously trained on only 8000 of 35549 focal recordings on CPU. Going to GPU lets us use ALL data + heavier augmentation + many more epochs without busting the 90-minute CPU **submission** budget at inference time.

## How to run
1. <https://www.kaggle.com/code/new>
2. Settings -> **Accelerator: GPU T4 x2** (or x1)
3. Internet: ON during training (off at submission, that's a different notebook)
4. Add Input -> the `birdclef-2026` competition
5. File -> Import Notebook -> upload this `.ipynb`
6. Save Version -> Save & Run All (Commit). Expect ~3-4 h.
7. Output `model_v50.keras` will be in `/kaggle/working/`. Add it to your `birdclef2026-model` Kaggle Dataset.

## What it does
- EfficientNetB0 (5.3M params, compatible with 90-min CPU inference)
- Input mel-spec (64, 626, 3) — same as current pipeline
- ALL 35k focal recordings preloaded once (~20 min) and cached on disk
- ALL 1.5k labelled soundscape windows + 5x sampling boost
- End-to-end training (no frozen phase) for 15 epochs with cosine LR
- Mixup + SpecAugment + label smoothing
- AdamW + weight decay
- Saves best model on held-out soundscape macro-AUC (group-by-file, seed 20260507 → SAME val as our local pipeline so directly comparable)

## Expected outcomes
- Local soundscape macro-AUC: 0.70 - 0.78
- Projected LB after CPU submission with TTA: 0.65 - 0.72

In [ ]:
import os, ast, time, json, math
import numpy as np
import pandas as pd
import librosa
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers
from sklearn.metrics import roc_auc_score
import warnings
warnings.filterwarnings('ignore')

print('TF', tf.__version__)
gpus = tf.config.list_physical_devices('GPU')
print('GPUs:', gpus)
for g in gpus:
    tf.config.experimental.set_memory_growth(g, True)
assert len(gpus) > 0, 'NO GPU. Settings -> Accelerator -> T4 GPU.'

In [ ]:
# =======================================================
# CONFIG  (matches CPU pipeline so model is interchangeable)
# =======================================================
DATA_DIR = '/kaggle/input/birdclef-2026'
TRAIN_CSV = f'{DATA_DIR}/train.csv'
TAXONOMY_CSV = f'{DATA_DIR}/taxonomy.csv'
TRAIN_AUDIO_DIR = f'{DATA_DIR}/train_audio'
TRAIN_SOUNDSCAPES_DIR = f'{DATA_DIR}/train_soundscapes'
TRAIN_SOUNDSCAPES_LABELS = f'{DATA_DIR}/train_soundscapes_labels.csv'
OUT_MODEL = '/kaggle/working/model_v50.keras'
OUT_LOG = '/kaggle/working/train_v50_log.json'
CACHE_DIR = '/kaggle/working/cache_v50'
os.makedirs(CACHE_DIR, exist_ok=True)

SAMPLE_RATE = 32000
DURATION = 5.0
N_MELS = 64
N_FFT = 2048
HOP_LENGTH = 256
FMIN = 20
FMAX = 16000
TOP_DB = 40.0
TARGET_HEIGHT = N_MELS
TARGET_WIDTH = 626

BACKBONE = 'EfficientNetB0'
BATCH_SIZE = 64
EPOCHS = 15
BASE_LR = 1e-3
WEIGHT_DECAY = 1e-5
LABEL_SMOOTHING = 0.05
MIXUP_ALPHA = 0.4
SOUNDSCAPE_BOOST = 5.0
RANDOM_SEED = 20260507  # MUST match our local val split for honest comparison
VAL_FRAC = 0.2

tf.keras.utils.set_random_seed(RANDOM_SEED)
rng_master = np.random.RandomState(RANDOM_SEED)

In [ ]:
# =======================================================
# AUDIO -> MEL  (matches experiment_template.make_melspec)
# =======================================================
def make_melspec(y, sr=SAMPLE_RATE):
    mel = librosa.feature.melspectrogram(
        y=y, sr=sr,
        n_mels=N_MELS, n_fft=N_FFT, hop_length=HOP_LENGTH,
        fmin=FMIN, fmax=FMAX, htk=True, norm='slaney',
    )
    mel = np.nan_to_num(mel, nan=0.0, posinf=0.0, neginf=0.0)
    mel_db = librosa.power_to_db(mel, ref=np.max, top_db=TOP_DB)
    mel_db = np.nan_to_num(mel_db, nan=-TOP_DB, posinf=0.0, neginf=-TOP_DB)
    norm = (mel_db + TOP_DB) / TOP_DB
    norm = np.clip(norm, 0.0, 1.0).astype(np.float32)
    if norm.shape[1] < TARGET_WIDTH:
        norm = np.pad(norm, ((0, 0), (0, TARGET_WIDTH - norm.shape[1])), mode='constant')
    elif norm.shape[1] > TARGET_WIDTH:
        norm = norm[:, :TARGET_WIDTH]
    return norm

In [ ]:
# =======================================================
# LABELS
# =======================================================
tax = pd.read_csv(TAXONOMY_CSV)
label_names = sorted(tax['primary_label'].unique().tolist())
n_classes = len(label_names)
label_to_idx = {l: i for i, l in enumerate(label_names)}
print('n_classes:', n_classes)

def focal_label_vec(primary, secondary):
    v = np.zeros(n_classes, dtype=np.float32)
    p = str(primary)
    if p in label_to_idx:
        v[label_to_idx[p]] = 1.0
    if isinstance(secondary, str) and secondary != '[]':
        try:
            for s in ast.literal_eval(secondary):
                s = str(s)
                if s in label_to_idx:
                    v[label_to_idx[s]] = 1.0
        except Exception:
            pass
    return v

def soundscape_label_vec(field):
    v = np.zeros(n_classes, dtype=np.float32)
    for tok in str(field).split(';'):
        tok = tok.strip()
        if tok in label_to_idx:
            v[label_to_idx[tok]] = 1.0
    return v

In [ ]:
# =======================================================
# PRECOMPUTE FOCAL CACHE (one-time, ~15 min on Kaggle)
# Uses parallel CPU on the GPU node.
# Saves at (N, 64, 626) float16 -> ~2.8 GB for 35k
# =======================================================
FOCAL_X = os.path.join(CACHE_DIR, 'focal_X.npy')
FOCAL_Y = os.path.join(CACHE_DIR, 'focal_Y.npy')

if os.path.exists(FOCAL_X) and os.path.exists(FOCAL_Y):
    Xf = np.load(FOCAL_X)
    Yf = np.load(FOCAL_Y)
    print(f'[focal cache hit] X={Xf.shape} Y={Yf.shape}')
else:
    print('Precomputing focal cache (~15-20 min)...')
    train_df = pd.read_csv(TRAIN_CSV)
    print(f'  recordings: {len(train_df)}')

    def process_one(args):
        path, primary, secondary = args
        try:
            y, _ = librosa.load(path, sr=SAMPLE_RATE, mono=True,
                                duration=DURATION)  # first 5 sec
            target = int(SAMPLE_RATE * DURATION)
            if len(y) < target:
                y = np.pad(y, (0, target - len(y)))
            elif len(y) > target:
                y = y[:target]
            mel = make_melspec(y).astype(np.float16)
            label = focal_label_vec(primary, secondary)
            return mel, label
        except Exception:
            return None

    args_list = []
    for row in train_df.itertuples(index=False):
        p = os.path.join(TRAIN_AUDIO_DIR, row.filename)
        if os.path.exists(p):
            args_list.append((p, row.primary_label, row.secondary_labels))

    from concurrent.futures import ThreadPoolExecutor
    Xf_list = []
    Yf_list = []
    t0 = time.time()
    with ThreadPoolExecutor(max_workers=8) as ex:
        for i, res in enumerate(ex.map(process_one, args_list), 1):
            if i % 1000 == 0:
                el = time.time() - t0
                rate = i / max(el, 1e-3)
                eta = (len(args_list) - i) / max(rate, 1e-3)
                print(f'  focal {i}/{len(args_list)}  {rate:.1f}/s  eta {eta/60:.1f}min')
            if res is None:
                continue
            Xf_list.append(res[0])
            Yf_list.append(res[1])

    Xf = np.stack(Xf_list, axis=0)
    Yf = np.stack(Yf_list, axis=0).astype(np.float32)
    print(f'  focal kept {len(Xf)} in {time.time()-t0:.0f}s')
    np.save(FOCAL_X, Xf)
    np.save(FOCAL_Y, Yf)

print('focal X mem:', Xf.nbytes / 1e9, 'GB')

In [ ]:
# =======================================================
# PRECOMPUTE SOUNDSCAPE CACHE (~3 min)
# =======================================================
SC_X = os.path.join(CACHE_DIR, 'sc_X.npy')
SC_Y = os.path.join(CACHE_DIR, 'sc_Y.npy')
SC_F = os.path.join(CACHE_DIR, 'sc_F.npy')

if all(os.path.exists(p) for p in [SC_X, SC_Y, SC_F]):
    Xs = np.load(SC_X)
    Ys = np.load(SC_Y)
    Fs = np.load(SC_F, allow_pickle=True)
    print(f'[soundscape cache hit] X={Xs.shape}')
else:
    sc_df = pd.read_csv(TRAIN_SOUNDSCAPES_LABELS)
    print(f'soundscape rows: {len(sc_df)}')
    Xs_list, Ys_list, Fs_list = [], [], []
    t0 = time.time()
    for i, row in enumerate(sc_df.itertuples(index=False), 1):
        if i % 200 == 0:
            print(f'  sc {i}/{len(sc_df)} ({(time.time()-t0):.0f}s)')
        p = os.path.join(TRAIN_SOUNDSCAPES_DIR, row.filename)
        if not os.path.exists(p):
            continue
        try:
            parts = str(row.start).split(':')
            ssec = int(parts[0])*3600 + int(parts[1])*60 + int(parts[2])
            y, _ = librosa.load(p, sr=SAMPLE_RATE, mono=True,
                                offset=ssec, duration=DURATION)
            target = int(SAMPLE_RATE * DURATION)
            if len(y) < target:
                y = np.pad(y, (0, target - len(y)))
            elif len(y) > target:
                y = y[:target]
            mel = make_melspec(y).astype(np.float16)
            Xs_list.append(mel)
            Ys_list.append(soundscape_label_vec(row.primary_label))
            Fs_list.append(row.filename)
        except Exception:
            continue
    Xs = np.stack(Xs_list, axis=0)
    Ys = np.stack(Ys_list, axis=0).astype(np.float32)
    Fs = np.array(Fs_list, dtype=object)
    print(f'soundscape kept {len(Xs)} in {time.time()-t0:.0f}s')
    np.save(SC_X, Xs)
    np.save(SC_Y, Ys)
    np.save(SC_F, Fs, allow_pickle=True)

# Group split by file
unique_files = np.unique(Fs)
n_val_files = max(1, int(round(len(unique_files) * VAL_FRAC)))
rng_split = np.random.RandomState(RANDOM_SEED)
val_files = set(rng_split.choice(unique_files, size=n_val_files, replace=False))
val_mask = np.array([f in val_files for f in Fs])
Xs_train, Ys_train = Xs[~val_mask], Ys[~val_mask]
Xs_val,   Ys_val   = Xs[val_mask],  Ys[val_mask]
print(f'soundscape train: {len(Xs_train)}  val: {len(Xs_val)}')

In [ ]:
# =======================================================
# COMBINED TRAINING POOL + tf.data PIPELINE
# =======================================================
X_train = np.concatenate([Xf, Xs_train], axis=0).astype(np.float16)
Y_train = np.concatenate([Yf, Ys_train], axis=0)
n_focal = len(Xf)
print(f'total train: {len(X_train)} ({n_focal} focal + {len(Xs_train)} sc)')

# Sample weights: rare-class boosted + soundscape boosted
class_pos = Y_train.sum(axis=0).clip(min=1)
inv = 1.0 / class_pos
inv = inv / inv.max()
sample_w = (Y_train * inv).max(axis=1)
boost = np.ones(len(X_train), dtype=np.float32)
boost[n_focal:] = SOUNDSCAPE_BOOST
sample_w = sample_w * boost
sample_w = np.where(sample_w > 0, sample_w, sample_w[sample_w > 0].mean())
sample_probs = sample_w / sample_w.sum()

# Generator with mixup + SpecAugment
rng_gen = np.random.RandomState(RANDOM_SEED + 1)
STEPS_PER_EPOCH = max(1, len(X_train) // BATCH_SIZE)

def generator():
    while True:
        idx = rng_gen.choice(len(X_train), size=BATCH_SIZE, replace=True, p=sample_probs)
        x = X_train[idx].astype(np.float32)
        y = Y_train[idx]
        # SpecAugment
        for i in range(BATCH_SIZE):
            for _ in range(2):
                fw = rng_gen.randint(2, 12)
                fs = rng_gen.randint(0, max(1, TARGET_HEIGHT - fw))
                x[i, fs:fs+fw, :] = 0.0
            for _ in range(2):
                tw = rng_gen.randint(5, 50)
                ts = rng_gen.randint(0, max(1, TARGET_WIDTH - tw))
                x[i, :, ts:ts+tw] = 0.0
        # 3 channels
        x = np.stack([x, x, x], axis=-1)
        # mixup
        if MIXUP_ALPHA > 0:
            lam = rng_gen.beta(MIXUP_ALPHA, MIXUP_ALPHA, size=BATCH_SIZE).astype(np.float32)
            lam = np.maximum(lam, 1.0 - lam)
            perm = rng_gen.permutation(BATCH_SIZE)
            lam_x = lam.reshape(-1, 1, 1, 1)
            lam_y = lam.reshape(-1, 1)
            x = lam_x * x + (1.0 - lam_x) * x[perm]
            y = lam_y * y + (1.0 - lam_y) * y[perm]
        yield x, y

ds = tf.data.Dataset.from_generator(
    generator,
    output_signature=(
        tf.TensorSpec(shape=(BATCH_SIZE, TARGET_HEIGHT, TARGET_WIDTH, 3), dtype=tf.float32),
        tf.TensorSpec(shape=(BATCH_SIZE, n_classes), dtype=tf.float32),
    ),
).prefetch(tf.data.AUTOTUNE)

print('steps per epoch:', STEPS_PER_EPOCH)

In [ ]:
# =======================================================
# MODEL  EfficientNetB0 (5.3M params, fast at CPU inference)
# =======================================================
def build_model():
    base = keras.applications.EfficientNetB0(
        weights='imagenet', include_top=False,
        input_shape=(TARGET_HEIGHT, TARGET_WIDTH, 3),
    )
    base.trainable = True
    inputs = keras.Input(shape=(TARGET_HEIGHT, TARGET_WIDTH, 3))
    x = base(inputs, training=True)
    x = layers.GlobalAveragePooling2D()(x)
    x = layers.BatchNormalization()(x)
    x = layers.Dropout(0.4)(x)
    x = layers.Dense(256, activation='relu')(x)
    x = layers.Dropout(0.3)(x)
    outputs = layers.Dense(n_classes, activation='sigmoid')(x)
    return keras.Model(inputs, outputs)

total_steps = STEPS_PER_EPOCH * EPOCHS
lr_schedule = keras.optimizers.schedules.CosineDecay(
    initial_learning_rate=BASE_LR,
    decay_steps=total_steps,
    alpha=0.05,
)
model = build_model()
model.compile(
    loss=keras.losses.BinaryCrossentropy(label_smoothing=LABEL_SMOOTHING),
    optimizer=keras.optimizers.AdamW(
        learning_rate=lr_schedule, weight_decay=WEIGHT_DECAY, clipnorm=1.0),
    metrics=[keras.metrics.AUC(name='auc')],
)
model.summary()

In [ ]:
# =======================================================
# TRAIN
# =======================================================
Xs_val_3 = np.stack([Xs_val.astype(np.float32)]*3, axis=-1)

def macro_auc(y_true, y_pred):
    aucs = []
    for c in range(y_true.shape[1]):
        if y_true[:, c].sum() == 0:
            continue
        try:
            aucs.append(roc_auc_score(y_true[:, c], y_pred[:, c]))
        except ValueError:
            continue
    return float(np.mean(aucs)) if aucs else float('nan')

best_auc = -np.inf
history = []

for epoch in range(1, EPOCHS + 1):
    print(f'\n=== epoch {epoch}/{EPOCHS} ===')
    t0 = time.time()
    model.fit(ds.take(STEPS_PER_EPOCH), epochs=1, verbose=1)
    preds = model.predict(Xs_val_3, batch_size=128, verbose=0)
    a = macro_auc(Ys_val, preds)
    print(f'  epoch {epoch}: soundscape macro-AUC = {a:.4f}  ({time.time()-t0:.0f}s)')
    history.append({'epoch': epoch, 'macro_auc': a})
    if a > best_auc:
        best_auc = a
        model.save(OUT_MODEL)
        print(f'  >> new best, saved {OUT_MODEL}')
    with open(OUT_LOG, 'w') as f:
        json.dump({
            'best_macro_auc': best_auc,
            'history': history,
            'config': {
                'backbone': BACKBONE,
                'n_mels': N_MELS,
                'epochs': EPOCHS,
                'batch_size': BATCH_SIZE,
                'mixup_alpha': MIXUP_ALPHA,
                'soundscape_boost': SOUNDSCAPE_BOOST,
                'label_smoothing': LABEL_SMOOTHING,
                'weight_decay': WEIGHT_DECAY,
                'random_seed': RANDOM_SEED,
                'n_focal': int(n_focal),
                'n_soundscape_train': int(len(Xs_train)),
                'n_soundscape_val': int(len(Xs_val)),
            },
        }, f, indent=2)

print(f'\n=== DONE: best macro-AUC = {best_auc:.4f} ===')
print(f'Model saved: {OUT_MODEL}')
print('Add this file to your `birdclef2026-model` Kaggle Dataset for the CPU submission notebook.')